<a href="https://colab.research.google.com/github/ancestor9/2026_Fall_Learning-Langchain-AI-Agent/blob/main/CH33_RAG_Query_Construction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **RAG Query Construction**

> **1. Text-to-Metadata Filter**

> **2. Text-to-SQL**


In [1]:
!pip -q install langchain langchain-classic langchain-community langchain-google-genai langchain-chroma sentence-transformers

In [2]:
# @title **Getting Set Up with LangChain**
import os
from google.colab import userdata

YOUR_API_KEY = userdata.get('Gemini-API-Key')
os.environ["GOOGLE_API_KEY"] = YOUR_API_KEY

In [3]:
# @title **환경 설정 (LLM & 임베딩)**
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.embeddings import HuggingFaceEmbeddings

llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash")
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

/tmp/ipykernel_13495/3567284554.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import HuggingFaceEmbeddings
/tmp/ipykernel_13495/3567284554.py:6: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

### **1. Text-to-Metadata Filter**

자연어 질문에서 검색 키워드뿐만 아니라 연도·장르 같은 조건(메타데이터)을 자동으로 추출해 필요한 데이터만 정밀하게 좁혀 검색하는 기술

with_structured_output으로 질문을 "주제어 + 연도 필터"로 나눕니다. 슬라이드의 year == 1980 구조적 조건을 직접 보여주기 위한 방식입니다

In [4]:
# @title **1. Text-to-Metadata Filter**

from typing import Optional
from pydantic import BaseModel, Field
from langchain_core.documents import Document
from langchain_chroma import Chroma

# 1) 본문 + 메타데이터(year)를 함께 임베딩해 저장
docs = [
    Document(page_content="A crew encounters a deadly alien creature in space.", metadata={"title": "Alien", "year": 1979}),
    Document(page_content="An astronaut-like hero battles alien emperor Ming to save Earth.", metadata={"title": "Flash Gordon", "year": 1980}),
    Document(page_content="A gentle alien is stranded on Earth and befriends a boy.", metadata={"title": "E.T.", "year": 1982}),
    Document(page_content="A boxer fights his own demons in and out of the ring.", metadata={"title": "Raging Bull", "year": 1980}),
]
vs = Chroma.from_documents(docs, embeddings)

# 2) 자연어 질문 -> (의미 검색어 + 메타데이터 필터)로 분해
class MovieQuery(BaseModel):
    topic: str = Field(description="임베딩 유사도 검색에 쓸 주제어")
    year: Optional[int] = Field(None, description="질문에 언급된 개봉 연도 (없으면 null)")

q = llm.with_structured_output(MovieQuery).invoke("What are movies about aliens in the year 1980?")

print(q)  # topic='aliens' year=1980

# 3) 유사도 검색 + 메타데이터 필터링
for d in vs.similarity_search(q.topic, k=3, filter={"year": q.year} if q.year else None):
    print(d.metadata, "-", d.page_content)

topic='aliens' year=1980
{'title': 'Flash Gordon', 'year': 1980} - An astronaut-like hero battles alien emperor Ming to save Earth.
{'year': 1980, 'title': 'Raging Bull'} - A boxer fights his own demons in and out of the ring.


### **Text-to-SQL**

사용자의 자연어 질문을 데이터베이스 조회용 SQL 쿼리로 자동 변환하여 데이터를 검색하는 기술

In [5]:
# @title **2. Text-to-SQL**
import sqlite3
from langchain_community.utilities import SQLDatabase
from langchain_classic.chains import create_sql_query_chain

# 1) 샘플 DB 생성
con = sqlite3.connect("users.db")
con.executescript("""
DROP TABLE IF EXISTS Users;
CREATE TABLE Users (Id INTEGER PRIMARY KEY, DisplayName TEXT, Location TEXT);
INSERT INTO Users VALUES (1,'Charlie','Boise, ID'),(2,'Alice','Boise, ID'),(3,'Bob','Seattle, WA');
""")
con.commit()
db = SQLDatabase.from_uri("sqlite:///users.db")

# 2) 질문 -> SQL 변환 (DB 스키마·dialect를 프롬프트에 자동 주입)
question = "Boise, ID에 사는 사용자를 DisplayName 순으로 보여줘"
sql = create_sql_query_chain(llm, db).invoke({"question": question})
sql = sql.replace("```sql", "").replace("```", "").replace("SQLQuery:", "").strip()
print(sql)

# 3) 안전장치: SELECT만 허용 (LLM이 만든 SQL을 그대로 실행하면 위험)
assert sql.lower().startswith("select"), "SELECT 이외의 쿼리는 실행하지 않습니다."

# 4) 실행 -> 결과를 LLM이 자연어 답변으로 변환
result = db.run(sql)
print(result)
print(llm.invoke(f"질문: {question}\nSQL: {sql}\n결과: {result}\n결과를 바탕으로 한국어로 답하세요.").content)

SELECT "DisplayName" FROM "Users" WHERE "Location" = 'Boise, ID' ORDER BY "DisplayName" ASC LIMIT 5;
[('Alice',), ('Charlie',)]
[{'type': 'text', 'text': 'Boise, ID에 사는 사용자는 DisplayName 순으로 **Alice**, **Charlie**입니다.', 'extras': {'signature': 'EugLCuULAWkUfRP1RwJ4UJOWlPWpgpnlREFdW639iIe0NUpi1DpQnsR4H6IyvHoezqrQLiqVSyr+Frt5iPbYJ3+5PqNcqz59LWipVTM3UL9sGdha9Ij1UsPvSQ9dUcmEpnnGcQp2GCTFrXKa6DhPtSEJLLJpVrgDyf2v7R3pFdtLOIBESgrfyRBPyxqNOpKilou+4XeSbHOsEQc8X3qD33QMcYScJWJx8+Gw2PlfnMtNASRotncjLg6BiRpz2ojn1AUMv+/mf4YC40kTnY9+CSIjAGINRIp5+Yz7cKmqMkLx1p8yodvbhhtkEfhtfHj6/0eMJMK7oBvab43NedUddbOLrQOw5XDXS3Bm5+pf6oTkqHyAfwNlymf//juPOPgk7qeL4hqZohSiAPattJrAzWZLAwZaGGyVv//e48CIZnocMu7TSE2w4aL7xhsY8EXRljdDfyJB/DZmXYd9n++/B9g0rlv292DttfSE6GdB5GFsFU1ohgNEVI2a23sXpvXg4Ygnkia+JLnCWDznMzjS8ff2h499ptjWyGjgSYpGUow5it2C+KnpTYK0yGG3nwcRUxCTj0U47hozJBYOThaWE+gu+DgoT4J6Ddz6FdmOhS3/6yOEtIrxDvdkkeSShNZTAtTHZ4jN1u8/M0V36MiCcE/I8DMARiLUyuqrq0k4bylGUNtggBny0/JZu3GYNbfBuUPs6LJZJzIPlgVjSRdMk9gYywJt879AH0Fd6iPC0XGRQdLo2VTc